# Temporal Preprocessing, Windowing, and Features From Scratch

## 1. Why Time-Series Preprocessing Is Different

Time-series preprocessing must preserve **temporal causality**.

The central rule is:

> Information from the future must never be used to prepare a training example from the past.

![Chronological split](images/03_temporal_split.png)

## 2. Learning Objectives

By the end of this notebook, you should be able to:

- explain temporal leakage,
- create chronological train, validation, and test splits,
- fit scaling parameters on training data only,
- create safe lag and rolling features,
- use calendar and cyclical features,
- distinguish lookback, horizon, and stride,
- build single-step windows manually,
- build multi-step windows,
- construct multivariate temporal inputs,
- understand future-covariate availability,
- inspect model-ready array shapes.

## 3. Load and Clean the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = "data/synthetic_hourly_demand.csv"

data = pd.read_csv(data_path)

data["timestamp"] = pd.to_datetime(data["timestamp"])

data = data.set_index("timestamp")

data["demand"] = data["demand"].interpolate(method="time")

print("Dataset shape:")
print(data.shape)

print()

print(data.head())

## 4. Temporal Data Leakage

Leakage happens when information unavailable at forecast time influences training.

Examples:

```text
Future target used as a feature                 ❌
Scaler fitted on train + validation + test      ❌
Random split before forecasting                 ❌
Centered rolling average using future points    ❌
```

## 5. Why Random Splitting Is Wrong

For forecasting, training should represent the past and testing should represent the future.

Correct conceptual order:

```text
Past → Train → Validation → Test → Future
```

## 6. Chronological Split

In [ ]:
number_of_rows = len(data)

train_end = int(number_of_rows * 0.70)

validation_end = int(number_of_rows * 0.85)

train_data = data.iloc[:train_end].copy()

validation_data = data.iloc[train_end:validation_end].copy()

test_data = data.iloc[validation_end:].copy()

print("Training rows:")
print(len(train_data))

print()

print("Validation rows:")
print(len(validation_data))

print()

print("Test rows:")
print(len(test_data))

In [ ]:
print("Training period:")
print(train_data.index.min())
print(train_data.index.max())

print()

print("Validation period:")
print(validation_data.index.min())
print(validation_data.index.max())

print()

print("Test period:")
print(test_data.index.min())
print(test_data.index.max())

## 7. Visualize the Split

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(train_data.index, train_data["demand"], label="Train")

plt.plot(validation_data.index, validation_data["demand"], label="Validation")

plt.plot(test_data.index, test_data["demand"], label="Test")

plt.xlabel("Time")
plt.ylabel("Demand")
plt.title("Chronological Train–Validation–Test Split")
plt.legend()

plt.show()

## 8. Scaling Without Leakage

Standardization uses:

\[
z = rac{x-\mu}{\sigma}
\]

The mean and standard deviation must be computed from **training data only**.

In [ ]:
train_mean = train_data["demand"].mean()

train_std = train_data["demand"].std()

print("Training mean:")
print(train_mean)

print()

print("Training standard deviation:")
print(train_std)

In [ ]:
train_scaled = (train_data["demand"] - train_mean) / train_std

validation_scaled = (validation_data["demand"] - train_mean) / train_std

test_scaled = (test_data["demand"] - train_mean) / train_std

print("Scaled training mean:")
print(train_scaled.mean())

print()

print("Scaled training standard deviation:")
print(train_scaled.std())

Test values can legitimately fall outside the training-scale range. That may reflect future distribution shift rather than an error.

## 9. Inverse Transformation

A scaled forecast can be converted back using:

\[
x = z\sigma+\mu
\]

In [ ]:
scaled_prediction = 1.25

original_prediction = scaled_prediction * train_std + train_mean

print("Scaled prediction:")
print(scaled_prediction)

print()

print("Original-unit prediction:")
print(original_prediction)

## 10. Create Lag Features

Valid forecasting features at time \(t\) can use earlier values such as:

```text
t - 1
t - 24
t - 168
```

In [ ]:
feature_data = data[["demand", "temperature", "humidity"]].copy()

feature_data["demand_lag_1"] = feature_data["demand"].shift(1)

feature_data["demand_lag_24"] = feature_data["demand"].shift(24)

feature_data["demand_lag_168"] = feature_data["demand"].shift(168)

print(feature_data.head(170))

## 11. Safe Rolling Features

For predicting time \(t\), a rolling feature should use values available **before** \(t\).

A safe pattern is:

```python
series.shift(1).rolling(window).mean()
```

In [ ]:
feature_data["rolling_mean_24"] = feature_data["demand"].shift(1).rolling(24).mean()

feature_data["rolling_std_24"] = feature_data["demand"].shift(1).rolling(24).std()

print(
    feature_data[
        ["demand", "rolling_mean_24", "rolling_std_24"]
    ].head(30)
)

## 12. Calendar Features

Temporal behavior may depend on:

- hour of day,
- day of week,
- weekend,
- month,
- holiday or operational schedule.

In [ ]:
feature_data["hour"] = feature_data.index.hour

feature_data["day_of_week"] = feature_data.index.dayofweek

feature_data["is_weekend"] = (feature_data["day_of_week"] >= 5).astype(int)

print(
    feature_data[
        ["hour", "day_of_week", "is_weekend"]
    ].head()
)

## 13. Cyclical Encoding

Hour 23 and hour 0 are adjacent in time but numerically far apart.

Use:

\[
sin\left(rac{2\pi h}{24}ight)
\]

and:

\[
cos\left(rac{2\pi h}{24}ight)
\]

In [ ]:
feature_data["hour_sin"] = np.sin(
    2.0 * np.pi * feature_data["hour"] / 24.0
)

feature_data["hour_cos"] = np.cos(
    2.0 * np.pi * feature_data["hour"] / 24.0
)

print(
    feature_data[
        ["hour", "hour_sin", "hour_cos"]
    ].head(25)
)

## 14. Sliding Windows

![Sliding window](images/04_sliding_window.png)

A forecasting model often receives a fixed past window and predicts one or more future values.

## 15. Lookback, Horizon, and Stride

### Lookback
Amount of historical context.

### Forecast horizon
How far into the future the model predicts.

### Stride
How far the window moves before the next sample is created.

## 16. Manual Single-Step Windowing

```text
Series:
10, 12, 14, 16, 18, 20

Lookback = 3

Input           Target
10 12 14   →      16
12 14 16   →      18
14 16 18   →      20
```

In [ ]:
series = np.array([10, 12, 14, 16, 18, 20], dtype=float)

lookback = 3

input_windows = []

targets = []

for start_index in range(0, len(series) - lookback):
    end_index = start_index + lookback

    input_window = series[start_index:end_index]

    target = series[end_index]

    input_windows.append(input_window)

    targets.append(target)

print("Input windows:")
print(input_windows)

print()

print("Targets:")
print(targets)

## 17. Convert to Arrays and Inspect Shapes

In [ ]:
X = np.array(input_windows)

y = np.array(targets)

print("X shape:")
print(X.shape)

print()

print("y shape:")
print(y.shape)

A dense model may use:

```text
(samples, lookback)
```

A sequence model commonly uses:

```text
(samples, timesteps, features)
```

## 18. Reusable Single-Step Window Function

In [ ]:
def create_single_step_windows(values, lookback):
    X = []
    y = []

    for start_index in range(0, len(values) - lookback):
        end_index = start_index + lookback

        input_window = values[start_index:end_index]

        target = values[end_index]

        X.append(input_window)

        y.append(target)

    X = np.array(X)

    y = np.array(y)

    return X, y

In [ ]:
train_values = train_scaled.to_numpy()

X_train, y_train = create_single_step_windows(
    train_values,
    lookback=24
)

print("X_train shape:")
print(X_train.shape)

print()

print("y_train shape:")
print(y_train.shape)

## 19. Add the Feature Dimension

In [ ]:
X_train_sequence = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

print("Sequence shape:")
print(X_train_sequence.shape)

## 20. Multi-Step Forecasting

```text
Lookback = 4
Horizon = 3

[10, 12, 14, 16] → [18, 20, 22]
```

In [ ]:
multi_series = np.array(
    [10, 12, 14, 16, 18, 20, 22, 24],
    dtype=float
)

lookback = 4

horizon = 3

inputs = []

targets = []

maximum_start = len(multi_series) - lookback - horizon + 1

for start_index in range(maximum_start):
    input_end = start_index + lookback

    target_end = input_end + horizon

    input_window = multi_series[start_index:input_end]

    target_window = multi_series[input_end:target_end]

    inputs.append(input_window)

    targets.append(target_window)

print("Inputs:")
print(inputs)

print()

print("Multi-step targets:")
print(targets)

## 21. Recursive vs Direct Multi-Step Forecasting

### Recursive

```text
predict t+1
use that prediction
predict t+2
...
```

Problem: errors can accumulate.

### Direct multi-output

```text
Past → [t+1, t+2, ..., t+H]
```

All horizons are predicted together.

## 22. Multivariate Forecasting

A multivariate input can contain multiple variables at each timestep:

```text
demand
temperature
humidity
```

Expected shape:

```text
(samples, timesteps, features)
```

In [ ]:
multivariate_values = train_data[
    ["demand", "temperature", "humidity"]
].to_numpy()

print("Raw multivariate shape:")
print(multivariate_values.shape)

In [ ]:
def create_multivariate_windows(values, target_column_index, lookback):
    X = []
    y = []

    for start_index in range(0, len(values) - lookback):
        end_index = start_index + lookback

        input_window = values[start_index:end_index, :]

        target = values[end_index, target_column_index]

        X.append(input_window)

        y.append(target)

    X = np.array(X)

    y = np.array(y)

    return X, y

In [ ]:
X_multi, y_multi = create_multivariate_windows(
    multivariate_values,
    target_column_index=0,
    lookback=24
)

print("Multivariate X shape:")
print(X_multi.shape)

print()

print("Multivariate y shape:")
print(y_multi.shape)

## 23. Future Covariate Availability

Suppose we forecast demand 24 hours ahead using temperature.

Two cases:

### Future temperature is genuinely known
For example, a weather forecast is available.

### Future actual temperature is taken directly from the test dataset
That is future information and can be leakage.

This distinction is crucial in multivariate forecasting.

## 24. Stride

Stride controls how far the window moves.

```text
stride = 1
```

creates highly overlapping samples.

Larger stride reduces overlap, number of samples, and computational cost.

## 25. Split-Boundary Context

The first validation target may legitimately use the last 24 training observations as historical context.

The important rule is:

> Historical input may cross the split boundary backward, but future target information must never cross into training.

This is why splitting the timeline first and then constructing target-aware windows is safer than windowing everything and randomly splitting later.

## 26. Final Model-Ready Shapes

A common deep-learning representation is:

```text
X:
(samples, lookback, features)

y single-step:
(samples,)

y multi-step:
(samples, horizon)
```

Always print these shapes before training.

## 27. Why This Notebook Matters

Many forecasting errors occur before model training:

```text
wrong split
leaky normalization
wrong target alignment
future covariate leakage
incorrect horizon
incorrect window offset
```

A sophisticated neural network cannot repair an invalid temporal pipeline.

## 28. Connection to the Next Notebook

We now know how to formulate valid temporal training examples.

The next notebook establishes:

```text
simple baselines
forecast metrics
residual analysis
rolling evaluation
uncertainty
```

These become the benchmark that later deep-learning models must beat.

## 29. Key Takeaways

- Forecasting data should be split chronologically.
- Preprocessing parameters must be learned from training data only.
- Lag and rolling features must use past information only.
- Lookback determines historical context.
- Horizon determines how far ahead we forecast.
- Windowing converts a temporal sequence into supervised-learning samples.
- Multivariate inputs add multiple features per timestep.
- Multi-step targets require careful target alignment.
- Future covariates are valid only if they are genuinely available at forecast time.